In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
%run ./schemas

In [0]:
def obtain_pokemon_url_list(lookup_table_name):
    evolution_chain_urls = spark.sql(f"""
        SELECT DISTINCT 
            evolution_chain.url AS url
        FROM 
            {STAGING_DATABASE_PREFIX}.species
    """)

    evolution_chain_url_list_total = [row['url'] for row in evolution_chain_urls.collect()]

    existing_url_list = check_existence(lookup_table_name, 'url')

    evo_chain_url_list = [url for url in evolution_chain_url_list_total if url not in existing_url_list]

    total_urls = len(evo_chain_url_list)

    return evo_chain_url_list, total_urls

In [0]:
LOOKUPS_TABLE_NAME = 'evo_chain_updated'

evolution_chain_url_list, total_urls = obtain_pokemon_url_list(LOOKUPS_TABLE_NAME)

for i, url in enumerate(evolution_chain_url_list):
    print(f'Starting {url}...')

    base_dict = {'evo_chain_url': url}

    evolution_chain_df = api_extraction(url, evo_chain_schema, base_dict)

    evolution_chain_df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.evo_chain")

    print(f'Completed {url} - {i+1}/{total_urls}')

    insert_query = f"""
        INSERT INTO {LOOKUPS_DATABASE_PREFIX}.{LOOKUPS_TABLE_NAME}
        VALUES ('{url}', '{datetime.now(timezone.utc).replace(tzinfo = None)}')
    """
    spark.sql(insert_query)